# 01 - Build Dataset
Carga snapshots y genera features + labels.

In [ ]:
from pathlib import Path

import pandas as pd

from ml.featureset import build_features
from ml.labeling import make_labels

SNAPSHOT_DIR = Path('data/snapshots')
files = list(SNAPSHOT_DIR.glob('*.parquet')) + list(SNAPSHOT_DIR.glob('*.csv'))
dfs = []
for path in files:
    if path.suffix == '.csv':
        dfs.append(pd.read_csv(path, parse_dates=True, index_col=0))
    else:
        dfs.append(pd.read_parquet(path))
df = pd.concat(dfs).sort_index() if dfs else pd.DataFrame()

if not df.empty:
    feature_set = build_features(df)
    labels = make_labels(df)
    dataset = feature_set.features.join(labels, how='inner')
    dataset.to_parquet('data/experiments/train.parquet')
    dataset.head()
else:
    print('No snapshots encontrados')
